# AROMA Step 1 — Complexity Analysis (Colab Guide)

**파이프라인 흐름:**
```
Dataset
  ↓ Phase 0  distribution_profiling.py  (기존 AROMA 저장소)
  ↓ Step 1   compute_complexity.py      (aroma 저장소)
       → complexity_report.json  (MCI, CCI, morphology_policy, context_policy)
```

**실행 전 확인:**
- Google Drive에 데이터셋 마운트 경로 확인
- 셀 2에서 저장소 URL 입력
- 셀 5에서 `DATASET_KEY` 선택

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 저장소 클론

> **TODO**: 아래 `YOUR_AROMA_REPO_URL` 을 실제 GitHub URL로 교체하세요.

In [ ]:
# Phase 0 스크립트 저장소 (distribution_profiling.py 포함)
AROMA_REPO = "YOUR_AROMA_REPO_URL"         # TODO: 입력

!git clone $AROMA_REPO /content/AROMA


## 3. 의존성 설치

In [ ]:
!pip install scikit-learn numpy pandas pyyaml pillow --quiet

## 4. 환경변수 설정

In [ ]:
import os

# ── AROMA (Phase 0 스크립트) ───────────────────────────────────────────────
os.environ['AROMA_REF']       = '/content/AROMA'

# ── AROMA (Step 1-4 스크립트) ────────────────────────────────────────
os.environ['AROMA_SCRIPTS']   = '/content/AROMA/scripts/aroma'
os.environ['DATASET_CONFIG']  = '/content/AROMA/dataset_config.json'

# ── Google Drive 출력 경로 ────────────────────────────────────────────────
os.environ['AROMA_DATA_BASE'] = '/content/drive/MyDrive/data/Aroma'
os.environ['AROMA_OUT']       = '/content/drive/MyDrive/data/Aroma/aroma_output'

# Python 편의 변수
AROMA_REF     = os.environ['AROMA_REF']
AROMA_SCRIPTS = os.environ['AROMA_SCRIPTS']
AROMA_OUT     = os.environ['AROMA_OUT']

print('환경변수 설정 완료')
for k in ['AROMA_REF', 'AROMA_SCRIPTS', 'AROMA_OUT', 'DATASET_CONFIG']:
    print(f'  {k} = {os.environ[k]}')

## 5. 데이터셋 키 선택

| 권장 | 도메인 | 설명 |
|------|--------|------|
| ✅ **isp_LSM_1** | ISP | 512px, 3678 train, 권장 시작점 |
| isp_LSM_2 | ISP | 512px |
| isp_ASM | ISP | 256px |
| mvtec_bottle | MVTec | — |
| visa_pcb4 | VisA | — |

In [ ]:
DATASET_KEY = 'isp_LSM_1'  # ← 변경 시 여기만 수정

os.environ['DATASET_KEY']    = DATASET_KEY
os.environ['PROFILING_DIR']  = f"{AROMA_OUT}/profiling/{DATASET_KEY}"
os.environ['COMPLEXITY_DIR'] = f"{AROMA_OUT}/complexity/{DATASET_KEY}"

print(f'DATASET_KEY    = {DATASET_KEY}')
print(f'PROFILING_DIR  = {os.environ["PROFILING_DIR"]}')
print(f'COMPLEXITY_DIR = {os.environ["COMPLEXITY_DIR"]}')

## 6. Phase 0 실행 — `distribution_profiling.py`

> 데이터셋당 1회 실행. 이미 실행했으면 이 셀은 건너뜀.

In [ ]:
!python $AROMA_REF/scripts/distribution_profiling.py \
    --dataset_config $DATASET_CONFIG \
    --dataset_key    $DATASET_KEY \
    --output_dir     $PROFILING_DIR

## 7. Phase 0 출력 확인

In [ ]:
import os, json

profiling_dir = os.environ['PROFILING_DIR']
expected_files = [
    'morphology_features.csv',
    'context_features.csv',
    'distribution_analysis.json',
    'morphology_clusters.json',
    'compatibility_matrix.json',
    'deficit_analysis.json',
]

print(f'Phase 0 출력 디렉토리: {profiling_dir}\n')
all_ok = True
for fname in expected_files:
    path = f'{profiling_dir}/{fname}'
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ 누락'
    print(f'  {status}  {fname}')
    if not exists:
        all_ok = False

if not all_ok:
    print('\n⚠️  누락 파일 있음 — Phase 0 셀(6번)을 먼저 실행하세요.')
else:
    print('\n✅ Phase 0 출력 정상. Step 1 진행 가능.')

## 8. Step 1 실행 — `compute_complexity.py`

In [ ]:
!python $AROMA_SCRIPTS/compute_complexity.py \
    --profiling_dir  $PROFILING_DIR \
    --output_dir     $COMPLEXITY_DIR \
    --weight_mode    equal \
    --local_staging

## 9. Step 1 결과 확인

In [ ]:
import json, os

report_path = f"{os.environ['COMPLEXITY_DIR']}/complexity_report.json"

if not os.path.exists(report_path):
    print('❌ complexity_report.json 없음 — Step 1 셀(8번)을 먼저 실행하세요.')
else:
    with open(report_path) as f:
        r = json.load(f)

    print('=== complexity_report.json ===')
    print(f"MCI               : {r['mci']:.4f}")
    print(f"CCI               : {r['cci']:.4f}")
    print(f"Morphology Policy : {r['morphology_policy']}")
    print(f"Context Policy    : {r['context_policy']}")
    print(f"Stability Margin  : {r['stability_margin']:.4f}")
    print(f"Weight Mode       : {r['weight_mode']}")
    print()
    print('--- MCI 구성 요소 ---')
    for k, v in r.get('mci_components', {}).get('raw', {}).items():
        norm_v = r['mci_components']['normalized'].get(k)
        norm_str = f"{norm_v:.4f}" if isinstance(norm_v, (int, float)) else "-.----"
        print(f"  {k:<20} raw={v:.4f}  norm={norm_str}")
    sil = r.get('mci_components', {}).get('silhouette_score')
    if sil is not None:
        print(f"  {'silhouette_score':<20} {sil:.4f}  (diagnostic)")
    print()
    print('--- CCI 구성 요소 ---')
    for k, v in r.get('cci_components', {}).get('raw', {}).items():
        norm_v = r['cci_components']['normalized'].get(k)
        norm_str = f"{norm_v:.4f}" if isinstance(norm_v, (int, float)) else "-.----"
        print(f"  {k:<20} raw={v:.4f}  norm={norm_str}")
    print()
    print('--- 후보 정책 평가 ---')
    for ev in r.get('evaluation_results', []):
        axis  = ev.get('axis', '?')
        pol   = ev.get('policy', '?')
        sil   = ev.get('silhouette', 0.0)
        stab  = ev.get('stability', '-')
        print(f"  [{axis:12s}] {pol:<14} silhouette={sil:.4f}  stability={stab}")

## 10. Ablation — Weight Mode 비교 (선택)

세 가지 weight mode를 모두 실행하고 MCI를 비교한다.

In [ ]:
import json, os

weight_modes = ['equal', 'entropy_heavy', 'cluster_heavy']
results = {}

for wm in weight_modes:
    out_dir = f"{os.environ['AROMA_OUT']}/complexity/{os.environ['DATASET_KEY']}_{wm}"
    os.environ['WM_OUT'] = out_dir
    os.environ['WM'] = wm
    print(f'\n=== weight_mode={wm} ===')
    !python $AROMA_SCRIPTS/compute_complexity.py \
        --profiling_dir  $PROFILING_DIR \
        --output_dir     $WM_OUT \
        --weight_mode    $WM
    rp = f'{out_dir}/complexity_report.json'
    if os.path.exists(rp):
        with open(rp) as f:
            r = json.load(f)
        results[wm] = {'mci': r['mci'], 'cci': r['cci'], 'policy': r['morphology_policy']}

print('\n=== Ablation 결과 요약 ===')
print(f'{"mode":<16} {"MCI":>8} {"CCI":>8} {"policy"}')
print('-' * 48)
for wm, v in results.items():
    print(f'{wm:<16} {v["mci"]:>8.4f} {v["cci"]:>8.4f} {v["policy"]}')

## 다음 단계

Step 1 완료 후 Step 2로 진행:

```python
!python $AROMA_SCRIPTS/prompt_generation.py \
    --profiling_dir  $PROFILING_DIR \
    --complexity_dir $COMPLEXITY_DIR \
    --output_dir     $AROMA_OUT/prompts/$DATASET_KEY
```

전체 파이프라인 실행 가이드: `notebooks/colab_pipeline_guide.ipynb` (예정)